# LODissea — Integrated Analysis of European Digital Cultural Heritage

This notebook consolidates three research questions about Europeana across six
countries: Italy, Germany, the Netherlands, Portugal, Spain, and France.

| RQ | Question |
|----|---------|
| **RQ1** | Does digital heritage volume reflect GLAM density or cultural expenditure (% GDP)? |
| **RQ2** | To what extent do countries meet Europeana's quality standards and how open are their datasets? |
| **RQ3** | What kind of institutions contribute to Europeana on behalf of each country? |

**Notebook structure:**
1. [Data Collection](#1-data-collection)
2. [Data Cleaning & Integration](#2-data-cleaning--integration)
3. [Visualizations & Analysis](#3-visualizations--analysis)

> **API key**: place your Europeana key in a `.env` file as `EUROPEANA_API_KEY=your_key`.
> All API calls are cached to CSV/JSON files — subsequent runs reuse cached data.


---
## 1. Data Collection <a id='1-data-collection'></a>

### 1.1 Install dependencies

In [ ]:
!pip install SPARQLWrapper rdflib eurostat plotly seaborn python-dotenv deep-translator

### 1.2 Imports & API key

The Europeana API key is loaded from a `.env` file (`EUROPEANA_API_KEY=...`). Get a free key at https://api.europeana.eu/.

In [ ]:
import os
import json
import re
import time
from pathlib import Path

import requests
import pandas as pd
import numpy as np
import eurostat
import plotly.express as px
import plotly.graph_objects as go
import matplotlib.pyplot as plt
from plotly.subplots import make_subplots

from SPARQLWrapper import SPARQLWrapper, JSON as SPARQL_JSON
from dotenv import load_dotenv
from deep_translator import GoogleTranslator

# ── API KEY: reads EUROPEANA_API_KEY from a local .env file ─────────────────
load_dotenv()
EUROPEANA_API_KEY = os.environ.get("EUROPEANA_API_KEY", "")
if not EUROPEANA_API_KEY:
    print("WARNING: EUROPEANA_API_KEY not set.\n"
          "Create a .env file with: EUROPEANA_API_KEY=your_key_here\n"
          "The notebook will fall back to cached CSV/JSON files where available.")

EUROPEANA_SEARCH_URL = "https://api.europeana.eu/record/v2/search.json"
DATA_DIR = Path("data")
DATA_DIR.mkdir(parents=True, exist_ok=True)
REQUEST_DELAY = 0.3


### 1.3 Country mapping

A single lookup table shared by all three research questions.

In [ ]:
COUNTRIES = {
    'IT': {'name_en': 'Italy',       'name_lower': 'italy',       'wd_id': 'Q38'},
    'DE': {'name_en': 'Germany',     'name_lower': 'germany',     'wd_id': 'Q183'},
    'NL': {'name_en': 'Netherlands', 'name_lower': 'netherlands', 'wd_id': 'Q55'},
    'PT': {'name_en': 'Portugal',    'name_lower': 'portugal',    'wd_id': 'Q45'},
    'ES': {'name_en': 'Spain',       'name_lower': 'spain',       'wd_id': 'Q29'},
    'FR': {'name_en': 'France',      'name_lower': 'france',      'wd_id': 'Q142'},
}

TARGET_COUNTRIES_LOWER = [info['name_lower'] for info in COUNTRIES.values()]
LANG_BY_COUNTRY = {
    'italy': 'it', 'france': 'fr', 'germany': 'de',
    'spain': 'es', 'netherlands': 'nl', 'portugal': 'pt',
}
ISO_FROM_LOWER = {info['name_lower']: iso for iso, info in COUNTRIES.items()}

df_countries = pd.DataFrame.from_dict(COUNTRIES, orient='index').reset_index()
df_countries.rename(columns={'index': 'iso_code'}, inplace=True)
df_countries


### 1.4 Europeana Search API helper

A single reusable wrapper with exponential-backoff retry on HTTP 429.

In [ ]:
def europeana_search(query="*", qf=None, facet=None, rows=0, **extra):
    """Wrapper around the Europeana Search API (v2)."""
    params = {"wskey": EUROPEANA_API_KEY, "query": query, "rows": rows, **extra}
    if qf:
        params["qf"] = qf
    if facet:
        params["facet"] = facet
        params["profile"] = "facets"

    for attempt in range(5):
        resp = requests.get(EUROPEANA_SEARCH_URL, params=params)
        if resp.status_code == 429:
            wait = 2 ** attempt
            print(f"Rate limited — waiting {wait}s...")
            time.sleep(wait)
            continue
        if resp.status_code != 200:
            print("URL:", resp.url)
            print("Status:", resp.status_code)
            resp.raise_for_status()
        time.sleep(REQUEST_DELAY)
        return resp.json()

    raise RuntimeError(f"Failed after retries: {params}")


if EUROPEANA_API_KEY:
    _test = europeana_search(query="*", qf=["COUNTRY:italy"], rows=1)
    print("Europeana API OK — total results for italy:", _test.get("totalResults"))
else:
    print("No API key — skipping connectivity test.")


### 1.5 Unified Europeana data collection

A **single API call per country** fetches everything needed across all three research
questions:

- `totalResults` → total item count (RQ1, RQ3)
- `DATA_PROVIDER` facet (paginated if needed) → full provider list (RQ1, RQ3) and
  top-20 for geo mapping (RQ2)
- `contentTier`, `metadataTier`, `RIGHTS`, `TYPE`, `proxy_dc_subject` facets → quality
  & openness analysis (RQ2)

Reusability counts (open/restricted/permission) still require three additional calls
per country because `reusability` is a request filter, not a facet.
All results are cached to `data/europeana_raw.json` — subsequent runs load from cache.

In [ ]:
# ── constants ────────────────────────────────────────────────────────────────
FACETS_ALL = "contentTier,metadataTier,RIGHTS,TYPE,proxy_dc_subject,DATA_PROVIDER"
REUSABILITY_VALUES = ["open", "restricted", "permission"]
RAW_FILE = DATA_DIR / "europeana_raw.json"
QUALITY_CONTENT_TIERS = {"2", "3", "4"}
QUALITY_METADATA_TIERS = {"A", "B", "C"}
PROVIDER_PAGE_SIZE = 500
RAW_PROVIDERS_DIR = DATA_DIR / "providers_data"
RAW_PROVIDERS_DIR.mkdir(parents=True, exist_ok=True)


def fetch_country_all(country_en):
    """Single call that fetches totalResults + all quality/openness facets
    + first page of DATA_PROVIDER (up to PROVIDER_PAGE_SIZE entries).

    Returns the raw JSON response dict.
    """
    params = {
        "wskey": EUROPEANA_API_KEY,
        "query": f'COUNTRY:"{country_en}"',
        "rows": 0,
        "facet": FACETS_ALL,
        "profile": "facets",
        "f.DATA_PROVIDER.facet.limit": PROVIDER_PAGE_SIZE,
    }
    r = requests.get(EUROPEANA_SEARCH_URL, params=params)
    r.raise_for_status()
    time.sleep(REQUEST_DELAY)
    return r.json()


def fetch_reusability(country_en):
    """Three additional calls (one per reusability value) for a country."""
    counts = {}
    for value in REUSABILITY_VALUES:
        r = requests.get(EUROPEANA_SEARCH_URL, params={
            "wskey": EUROPEANA_API_KEY,
            "query": f'COUNTRY:"{country_en}"',
            "reusability": value, "rows": 0,
        })
        r.raise_for_status()
        counts[value] = r.json().get("totalResults", 0)
        time.sleep(REQUEST_DELAY)
    return counts


def fetch_extra_providers(country_lower, first_page_count):
    """Fetch additional provider pages when a country has >PROVIDER_PAGE_SIZE providers."""
    all_rows = []
    offset = PROVIDER_PAGE_SIZE
    for _ in range(50):
        result = europeana_search(
            query="*",
            qf=[f"COUNTRY:{country_lower}"],
            facet="DATA_PROVIDER",
            rows=0,
            **{
                "f.DATA_PROVIDER.facet.limit": PROVIDER_PAGE_SIZE,
                "f.DATA_PROVIDER.facet.offset": offset,
            },
        )
        page_rows = [
            {"country": country_lower, "provider": f["label"], "count": f["count"]}
            for facet in result.get("facets", [])
            for f in facet.get("fields", [])
        ]
        all_rows.extend(page_rows)
        if len(page_rows) < PROVIDER_PAGE_SIZE:
            break
        offset += PROVIDER_PAGE_SIZE
    return all_rows


# ── Run or load from cache ────────────────────────────────────────────────────
if EUROPEANA_API_KEY and not RAW_FILE.exists():
    raw_results = {}
    reusability_results = {}

    for iso, info in COUNTRIES.items():
        country_en = info['name_en']
        country_lower = info['name_lower']
        print(f"Fetching {country_en}...", end=" ")

        data = fetch_country_all(country_en)
        raw_results[country_en] = data
        reusability_results[country_en] = fetch_reusability(country_en)

        # Extract first-page providers from the master response
        dp_facet = next((f for f in data.get('facets', []) if f['name'] == 'DATA_PROVIDER'), None)
        first_page = [
            {"country": country_lower, "provider": f["label"], "count": f["count"]}
            for f in (dp_facet['fields'] if dp_facet else [])
        ]
        # Paginate if the first page was full (likely more providers exist)
        extra = fetch_extra_providers(country_lower, len(first_page)) if len(first_page) >= PROVIDER_PAGE_SIZE else []
        all_providers = first_page + extra

        country_df = pd.DataFrame(all_providers)
        country_df.to_csv(RAW_PROVIDERS_DIR / f"providers_{country_lower}.csv", index=False)
        print(f"{data.get('totalResults'):,} items, {len(all_providers)} providers")

    # Store reusability inside raw_results for convenience
    for country_en, counts in reusability_results.items():
        raw_results[country_en]["reusability"] = counts

    with open(RAW_FILE, "w") as f:
        json.dump(raw_results, f, indent=2)
    print(f"\nSaved {RAW_FILE}")

else:
    if RAW_FILE.exists():
        with open(RAW_FILE) as f:
            raw_results = json.load(f)
        print(f"Loaded cached {RAW_FILE} ({len(raw_results)} countries)")
    else:
        raw_results = {}
        print("WARNING: No API key and no cached file found. Set EUROPEANA_API_KEY.")

    # Load reusability from cache (stored inside raw_results) or skip if missing
    reusability_results = {
        country: data["reusability"]
        for country, data in raw_results.items()
        if "reusability" in data
    }
    if not reusability_results:
        print("NOTE: Reusability data not in cache — re-run with API key to collect it.")


# ── Derive shared DataFrames from the unified raw_results ─────────────────────
# totals_df / df_europeana (for RQ1 + RQ3)
totals_df = pd.DataFrame([
    {"country": info['name_lower'], "total_items": raw_results.get(info['name_en'], {}).get("totalResults", 0)}
    for info in COUNTRIES.values()
])
totals_df.to_csv(DATA_DIR / "tot_items_by_country.csv", index=False)

df_europeana = (
    totals_df
    .assign(iso_code=lambda d: d['country'].map(ISO_FROM_LOWER))
    .rename(columns={'total_items': 'europeana_total'})
    [['iso_code', 'europeana_total']]
    .sort_values('europeana_total', ascending=False, ignore_index=True)
)

# providers_by_country (for RQ1 count + RQ3 full list)
providers_by_country = {}
for country_lower in TARGET_COUNTRIES_LOWER:
    p = RAW_PROVIDERS_DIR / f"providers_{country_lower}.csv"
    if p.exists():
        providers_by_country[country_lower] = pd.read_csv(p)
    else:
        # Fall back to first-page providers from raw_results
        country_en = next(info['name_en'] for info in COUNTRIES.values() if info['name_lower'] == country_lower)
        dp_facet = next(
            (f for f in raw_results.get(country_en, {}).get('facets', []) if f['name'] == 'DATA_PROVIDER'),
            None
        )
        rows = [
            {"country": country_lower, "provider": f["label"], "count": f["count"]}
            for f in (dp_facet['fields'] if dp_facet else [])
        ]
        providers_by_country[country_lower] = pd.DataFrame(rows)

providers_df_all = pd.concat(providers_by_country.values(), ignore_index=True)

# df_providers: RQ1 provider count per country
df_providers = pd.DataFrame([
    {'iso_code': ISO_FROM_LOWER[c], 'europeana_providers': len(df)}
    for c, df in providers_by_country.items()
])

print("\nItem counts per country:")
display(df_europeana)
print("\nProvider counts per country:")
display(df_providers)


*Item counts include Tier 0 entries which are indexed but not visible on the Europeana website.*

### 1.6 Public cultural expenditure — Eurostat COFOG

General government expenditure on recreation, culture and religion (COFOG GF08) as a percentage of GDP (2022), from Eurostat table `gov_10a_exp`.

In [ ]:
df_exp = eurostat.get_data_df('gov_10a_exp')

df_exp_filtered = df_exp[
    (df_exp['sector'] == 'S13') &
    (df_exp['unit'] == 'PC_GDP') &
    (df_exp['cofog99'] == 'GF08') &
    (df_exp['na_item'] == 'TE') &
    (df_exp['geo\\TIME_PERIOD'].isin(COUNTRIES.keys()))
][['geo\\TIME_PERIOD', '2022']].rename(
    columns={'geo\\TIME_PERIOD': 'iso_code', '2022': 'culture_expenditure_gpd_2022'}
)
df_exp_filtered


### 1.7 GLAM institution count per country — Wikidata SPARQL

Counts museums (Q33506), libraries (Q7075), archives (Q166118), and galleries (Q1007870) per country. Queried in batches of 3 countries.

In [ ]:
sparql_wd = SPARQLWrapper("https://query.wikidata.org/sparql")
sparql_wd.agent = "info-viz-student-project/1.0 (mailto:martina.uccheddu@studio.unibo.it)"
sparql_wd.setReturnFormat(SPARQL_JSON)

countries_list = [(iso, info['wd_id'], info['name_en']) for iso, info in COUNTRIES.items()]
BATCH_SIZE = 3
wikidata_rows = []


def create_batch(lst, dimension):
    for i in range(0, len(lst), dimension):
        yield lst[i:i + dimension]


for idx_batch, batch in enumerate(create_batch(countries_list, BATCH_SIZE), start=1):
    qid_vals = " ".join(
        f"wd:{p[1].split('/')[-1].replace('wd:', '')}" for p in batch
    )
    query = f"""
    SELECT ?country (COUNT(DISTINCT ?item) AS ?count) WHERE {{
      VALUES ?country {{ {qid_vals} }}
      VALUES ?type {{ wd:Q33506 wd:Q7075 wd:Q166118 wd:Q1007870 }}
      ?item wdt:P31 ?type ; wdt:P17 ?country .
    }}
    GROUP BY ?country
    """
    sparql_wd.setQuery(query)
    try:
        results = sparql_wd.query().convert()
        count_temp = {
            row["country"]["value"].split("/")[-1]: int(row["count"]["value"])
            for row in results["results"]["bindings"]
        }
        for iso, qid, name in batch:
            cqid = qid.split("/")[-1].replace("wd:", "")
            wikidata_rows.append({'iso_code': iso, 'glam_count_wikidata': count_temp.get(cqid, 0)})
    except Exception as e:
        print(f"Batch {idx_batch} error: {e}")
        for iso, qid, name in batch:
            wikidata_rows.append({'iso_code': iso, 'glam_count_wikidata': 0})

    if idx_batch * BATCH_SIZE < len(countries_list):
        time.sleep(1.5)

df_wikidata = pd.DataFrame(wikidata_rows)
df_wikidata


### 1.8 Wikidata entity resolution for providers — RQ3

For each distinct provider in the top-100 per country, searches Wikidata to obtain a QID and description (English first; country language as fallback). Results are cached to `data/wikidata_resolution.csv` — subsequent runs load from cache and skip all API calls.

In [ ]:
WD_API_URL = "https://www.wikidata.org/w/api.php"
WD_HEADERS = {"User-Agent": "InfoVis-course-project/0.1 (student project)"}
WIKIDATA_SPARQL_URL = "https://query.wikidata.org/sparql"
WD_RESOLUTION_CSV = DATA_DIR / "wikidata_resolution.csv"


def search_wikidata_rest(name, lang="en", max_retries=3):
    params = {
        "action": "wbsearchentities", "search": name, "language": lang,
        "format": "json", "limit": 1, "type": "item",
    }
    for attempt in range(max_retries):
        try:
            resp = requests.get(WD_API_URL, params=params, headers=WD_HEADERS, timeout=10)
        except requests.exceptions.RequestException as e:
            print(f"network error searching '{name}': {e}")
            time.sleep(2 ** attempt)
            continue
        if resp.status_code == 200:
            results = resp.json().get("search", [])
            if not results:
                return {"qid": None, "description": None, "desc_lang": None, "status": "no_match"}
            top = results[0]
            display_desc = top.get("display", {}).get("description", {})
            description = display_desc.get("value") or top.get("description")
            actual_lang = display_desc.get("language")
            return {"qid": top["id"], "description": description,
                    "desc_lang": actual_lang if description else None, "status": "ok"}
        if resp.status_code in (429, 502, 503):
            time.sleep(2 ** attempt)
            continue
        return {"qid": None, "description": None, "desc_lang": None,
                "status": f"http_{resp.status_code}"}
    return {"qid": None, "description": None, "desc_lang": None, "status": "retries_exhausted"}


def resolve_provider_wikidata(name, country):
    result = search_wikidata_rest(name, lang="en")
    if result["status"] == "ok" and result["description"]:
        result["resolution_lang"] = "en"
        return result
    country_lang = LANG_BY_COUNTRY.get(country, "en")
    local_result = search_wikidata_rest(name, lang=country_lang)
    if local_result["status"] == "ok" and (local_result["description"] or not result["qid"]):
        local_result["resolution_lang"] = country_lang
        return local_result
    if result["status"] == "ok":
        result["resolution_lang"] = "en"
        return result
    local_result["resolution_lang"] = country_lang
    return local_result


# ── top-100 per country (needed to know which providers to resolve) ───────────
def coverage_at_n(country_df, n):
    s = country_df.sort_values("count", ascending=False)
    return s["count"].head(n).sum() / s["count"].sum()

TOP_N_PER_COUNTRY = 100
top_providers_df = pd.concat(
    [df.sort_values("count", ascending=False).head(TOP_N_PER_COUNTRY)
     for df in providers_by_country.values()],
    ignore_index=True
)
distinct_providers = top_providers_df[["provider", "country"]].drop_duplicates()


# ── Load from cache or run resolution ─────────────────────────────────────────
if WD_RESOLUTION_CSV.exists():
    wd_df = pd.read_csv(WD_RESOLUTION_CSV)
    wikidata_resolution = {
        row["provider"]: {
            "qid": row["qid"] if pd.notna(row["qid"]) else None,
            "description": row["description"] if pd.notna(row["description"]) else None,
            "description_en": row["description_en"] if pd.notna(row["description_en"]) else None,
            "desc_lang": row["desc_lang"] if pd.notna(row["desc_lang"]) else None,
            "status": row["status"],
            "resolution_lang": row["resolution_lang"] if pd.notna(row["resolution_lang"]) else None,
        }
        for _, row in wd_df.iterrows()
    }
    print(f"Loaded {len(wikidata_resolution)} cached Wikidata resolutions from {WD_RESOLUTION_CSV}")
else:
    wikidata_resolution = {}
    for _, row in distinct_providers.iterrows():
        name, country = row["provider"], row["country"]
        wikidata_resolution[name] = resolve_provider_wikidata(name, country)
        time.sleep(0.3)

    wd_resolved = sum(1 for r in wikidata_resolution.values() if r["status"] == "ok")
    print(f"{wd_resolved} / {len(wikidata_resolution)} providers resolved via Wikidata")

    # Save to CSV for future runs
    pd.DataFrame([
        {"provider": name, **info}
        for name, info in wikidata_resolution.items()
    ]).to_csv(WD_RESOLUTION_CSV, index=False)
    print(f"Saved to {WD_RESOLUTION_CSV}")


### 1.9 Translate non-English Wikidata descriptions

Descriptions retrieved in non-English languages are translated to English so that the classification rules (section 2.3) can match them. Results are stored in the `description_en` column of `wikidata_resolution.csv`.

In [ ]:
translation_cache = {}


def translate_to_english(text, src_lang):
    if not text:
        return None
    if src_lang == "en":
        return text
    key = (src_lang, text)
    if key in translation_cache:
        return translation_cache[key]
    try:
        translated = GoogleTranslator(source=src_lang, target="en").translate(text)
    except Exception as e:
        print(f"translation failed for lang={src_lang}: {e}")
        translated = None
    translation_cache[key] = translated
    time.sleep(0.2)
    return translated


# Only translate if description_en is missing from cache
needs_translation = [
    name for name, info in wikidata_resolution.items()
    if not info.get("description_en") and info.get("description")
]
if needs_translation:
    print(f"Translating {len(needs_translation)} descriptions...")
    for name in needs_translation:
        info = wikidata_resolution[name]
        info["description_en"] = translate_to_english(info.get("description"), info.get("desc_lang"))
    # Re-save CSV with translations filled in
    pd.DataFrame([
        {"provider": name, **info}
        for name, info in wikidata_resolution.items()
    ]).to_csv(WD_RESOLUTION_CSV, index=False)
    print(f"Updated {WD_RESOLUTION_CSV}")
else:
    print("All descriptions already translated (loaded from cache).")


### 1.10 SPARQL fallback — `instance_of` / `field_of_work` — RQ3

For providers where the Wikidata text description alone is not enough, a SPARQL query fetches P31 (instance of) and P101 (field of work) in batch. Results are cached to `data/wikidata_sparql_info.csv`.

In [ ]:
SPARQL_INFO_CSV = DATA_DIR / "wikidata_sparql_info.csv"


def fetch_institution_info(qids, batch_size=50):
    results = {}
    qids = [q for q in qids if q]
    for i in range(0, len(qids), batch_size):
        batch = qids[i:i + batch_size]
        values = " ".join(f"wd:{q}" for q in batch)
        query = f"""
        SELECT ?item ?instanceOfLabel ?fieldOfWorkLabel WHERE {{
          VALUES ?item {{ {values} }}
          OPTIONAL {{ ?item wdt:P31 ?instanceOf . ?instanceOf rdfs:label ?instanceOfLabel . FILTER(LANG(?instanceOfLabel) = "en") }}
          OPTIONAL {{ ?item wdt:P101 ?fieldOfWork . ?fieldOfWork rdfs:label ?fieldOfWorkLabel . FILTER(LANG(?fieldOfWorkLabel) = "en") }}
        }}
        """
        resp = requests.get(
            WIKIDATA_SPARQL_URL,
            params={"query": query, "format": "json"},
            headers=WD_HEADERS
        )
        if resp.status_code != 200:
            continue
        for row in resp.json()["results"]["bindings"]:
            qid = row["item"]["value"].rsplit("/", 1)[-1]
            entry = results.setdefault(qid, {"instance_of": [], "field_of_work": []})
            if "instanceOfLabel" in row:
                entry["instance_of"].append(row["instanceOfLabel"]["value"])
            if "fieldOfWorkLabel" in row:
                entry["field_of_work"].append(row["fieldOfWorkLabel"]["value"])
        time.sleep(0.5)
    return results


if SPARQL_INFO_CSV.exists():
    sparql_df = pd.read_csv(SPARQL_INFO_CSV)
    sparql_info = {}
    for _, row in sparql_df.iterrows():
        qid = row["qid"]
        entry = sparql_info.setdefault(qid, {"instance_of": [], "field_of_work": []})
        if pd.notna(row.get("instance_of")) and row["instance_of"]:
            entry["instance_of"] = row["instance_of"].split("|") if isinstance(row["instance_of"], str) else []
        if pd.notna(row.get("field_of_work")) and row["field_of_work"]:
            entry["field_of_work"] = row["field_of_work"].split("|") if isinstance(row["field_of_work"], str) else []
    print(f"Loaded SPARQL info for {len(sparql_info)} providers from {SPARQL_INFO_CSV}")
else:
    qid_list = [info["qid"] for info in wikidata_resolution.values() if info.get("qid")]
    sparql_info = fetch_institution_info(qid_list)
    print(f"SPARQL instance_of/field_of_work found for {len(sparql_info)} / {len(qid_list)} QIDs")

    # Save to CSV (pipe-delimited lists to preserve multiple values)
    sparql_rows = [
        {
            "qid": qid,
            "instance_of": "|".join(info["instance_of"]) if info["instance_of"] else "",
            "field_of_work": "|".join(info["field_of_work"]) if info["field_of_work"] else "",
        }
        for qid, info in sparql_info.items()
    ]
    pd.DataFrame(sparql_rows).to_csv(SPARQL_INFO_CSV, index=False)
    print(f"Saved to {SPARQL_INFO_CSV}")


### 1.11 Geographic coordinates for top-20 providers — RQ2

Resolves coordinates through a cascade: Europeana entity API → Wikidata by QID → Wikidata by name → manual fallback. Results are cached to `data/map.csv`.

In [ ]:
MAP_CSV = DATA_DIR / "map.csv"
ENTITY_SUGGEST_URL = "https://api.europeana.eu/entity/suggest"


def facet_dict(country_data, facet_name):
    """Extract a facet from a raw_results entry as {label: count}."""
    for facet in country_data.get("facets", []):
        if facet["name"] == facet_name:
            return {f["label"]: f["count"] for f in facet["fields"]}
    return {}


def get_top_n_providers(raw, n=20):
    rows = []
    for country, data in raw.items():
        dp = facet_dict(data, "DATA_PROVIDER")
        for name, count in sorted(dp.items(), key=lambda x: -x[1])[:n]:
            rows.append({"country": country, "provider": name, "count": count})
    return pd.DataFrame(rows)


def lookup_europeana_entity(provider_name):
    params = {
        "wskey": EUROPEANA_API_KEY, "text": provider_name,
        "type": "Organization", "fl": "id,prefLabel,owl:sameAs,geo:lat,geo:long",
    }
    try:
        r = requests.get(ENTITY_SUGGEST_URL, params=params, timeout=10)
        if r.status_code == 200:
            items = r.json().get("items", [])
            if items:
                item = items[0]
                lat = item.get("geo:lat")
                lon = item.get("geo:long")
                wikidata_id = next(
                    (uri.split("/")[-1] for uri in (item.get("owl:sameAs") or []) if "wikidata" in uri),
                    None
                )
                return {
                    "matched": True, "has_geo": lat is not None and lon is not None,
                    "latitude": float(lat) if lat else None,
                    "longitude": float(lon) if lon else None,
                    "wikidata_id": wikidata_id,
                }
    except Exception as e:
        print(f"Entity lookup failed for '{provider_name}': {e}")
    return {"matched": False, "has_geo": False, "latitude": None, "longitude": None, "wikidata_id": None}


def get_wikidata_coords(qid_or_name, by="qid"):
    if by == "name":
        params = {"action": "wbsearchentities", "search": qid_or_name,
                  "language": "en", "format": "json", "limit": 1, "type": "item"}
        r = requests.get(WD_API_URL, params=params, headers=WD_HEADERS, timeout=10)
        results = r.json().get("search", []) if r.status_code == 200 else []
        if not results:
            return None, None
        qid_or_name = results[0]["id"]
    params = {"action": "wbgetentities", "ids": qid_or_name, "props": "claims", "format": "json"}
    try:
        r = requests.get(WD_API_URL, params=params, headers=WD_HEADERS, timeout=10)
        claims = r.json().get("entities", {}).get(qid_or_name, {}).get("claims", {})
        if "P625" in claims:
            coord = claims["P625"][0]["mainsnak"]["datavalue"]["value"]
            return coord.get("latitude"), coord.get("longitude")
    except Exception as e:
        print(f"Wikidata coord lookup failed: {e}")
    return None, None


MANUAL_GEO = [
    {"country": "Italy", "provider": "Central Museum of the Risorgimento", "count": 53211,
     "latitude": 41.89413575264878, "longitude": 12.483798667349626, "source": "manual"},
    {"country": "France", "provider": "Palais Galliera - Musée de la Mode de la Ville de Paris", "count": 44495,
     "latitude": 48.86608033629242, "longitude": 2.2965618523973155, "source": "manual"},
    {"country": "Italy", "provider": "Experimental Cinematography Center", "count": 40538,
     "latitude": 41.851058155342265, "longitude": 12.569479552005237, "source": "manual"},
    {"country": "Italy", "provider": "Epigraphic Dabatase Bari", "count": 40283,
     "latitude": 41.12112736784277, "longitude": 16.868604802773277, "source": "manual"},
    {"country": "Italy", "provider": "Marciana National Library", "count": 29597,
     "latitude": 45.43353428986529, "longitude": 12.339422752198944, "source": "manual"},
    {"country": "Italy", "provider": "Turin Gallery for Modern and Contemporary Art", "count": 29395,
     "latitude": 45.065010771493675, "longitude": 7.66921379635602, "source": "manual"},
    {"country": "France", "provider": "National and University Library of Strasbourg", "count": 21320,
     "latitude": 48.5872587063354, "longitude": 7.755901183065034, "source": "manual"},
    {"country": "Italy", "provider": "Library of the S. Pietro a Majella Conservatory", "count": 20154,
     "latitude": 40.849630767959596, "longitude": 14.252446682637897, "source": "manual"},
    {"country": "Italy", "provider": "Provincial Library Magna Capitana", "count": 13065,
     "latitude": 41.45671868731526, "longitude": 15.558523953833472, "source": "manual"},
    {"country": "Italy", "provider": "Estense University Library", "count": 12372,
     "latitude": 44.64842579381791, "longitude": 10.920992367497412, "source": "manual"},
    {"country": "France", "provider": "Rhône-Alpes Laboratory for Historical Research", "count": 10046,
     "latitude": 45.7334134679147, "longitude": 4.833417509886817, "source": "manual"},
    {"country": "Portugal", "provider": "MUDE – Museu do Design", "count": 1927,
     "latitude": 38.70924101252115, "longitude": -9.136938317468571, "source": "manual"},
    {"country": "Portugal", "provider": "Fernando Pessoa's House", "count": 1210,
     "latitude": 38.716820396673036, "longitude": -9.162572105823484, "source": "manual"},
    {"country": "Portugal", "provider": "Lisbon's Film & Theatre School", "count": 774,
     "latitude": 38.77728867945194, "longitude": -9.234926049524557, "source": "manual"},
    {"country": "Portugal", "provider": "Cinemateca Portuguesa - Museu do cinema", "count": 654,
     "latitude": 38.721224635535016, "longitude": -9.14865910212588, "source": "manual"},
]


if MAP_CSV.exists():
    provider_geo_final = pd.read_csv(MAP_CSV)
    print(f"Loaded {len(provider_geo_final)} geo-resolved providers from {MAP_CSV}")
elif raw_results and EUROPEANA_API_KEY:
    top20_df = get_top_n_providers(raw_results, n=20)

    # Step 1: Europeana entity lookup
    geo_rows = []
    for _, row in top20_df.iterrows():
        result = lookup_europeana_entity(row["provider"])
        geo_rows.append({**row.to_dict(), **result})
        time.sleep(0.2)
    geo_df = pd.DataFrame(geo_rows)

    # Step 2: Wikidata by QID for matched-but-no-geo
    qid_rows = []
    for _, row in geo_df[
        geo_df["matched"] & ~geo_df["has_geo"].fillna(False) & geo_df["wikidata_id"].notna()
    ].iterrows():
        lat, lon = get_wikidata_coords(row["wikidata_id"], by="qid")
        if lat and lon:
            qid_rows.append({"country": row["country"], "provider": row["provider"],
                              "count": row["count"], "latitude": lat, "longitude": lon,
                              "source": "wikidata_qid"})
        time.sleep(0.3)
    fallback_qid = pd.DataFrame(qid_rows) if qid_rows else pd.DataFrame()

    # Step 3: Wikidata by name for still-unresolved
    resolved_so_far = set(geo_df[geo_df["has_geo"].fillna(False)]["provider"]) | (
        set(fallback_qid["provider"]) if not fallback_qid.empty else set())
    name_rows = []
    for _, row in geo_df[~geo_df["provider"].isin(resolved_so_far)].iterrows():
        lat, lon = get_wikidata_coords(row["provider"], by="name")
        if lat and lon:
            name_rows.append({"country": row["country"], "provider": row["provider"],
                               "count": row["count"], "latitude": lat, "longitude": lon,
                               "source": "wikidata_name"})
        time.sleep(0.3)
    fallback_name = pd.DataFrame(name_rows) if name_rows else pd.DataFrame()

    # Step 4: Consolidate + manual fallback
    europeana_geo = geo_df[geo_df["has_geo"].fillna(False)][
        ["country", "provider", "count", "latitude", "longitude"]
    ].assign(source="europeana_entity")

    all_frames = [df for df in [europeana_geo, fallback_qid, fallback_name, pd.DataFrame(MANUAL_GEO)]
                  if df is not None and not df.empty]
    provider_geo_final = (
        pd.concat(all_frames, ignore_index=True)
        .drop_duplicates(subset=["provider"], keep="first")
    )

    # Correct France 'Ministry of Culture' (automated lookup returned wrong country)
    mask = (
        provider_geo_final["provider"].str.contains("Ministry of Culture", case=False) &
        (provider_geo_final["country"] == "France")
    )
    provider_geo_final.loc[mask, ["latitude", "longitude"]] = [48.86255840307406, 2.3388283365026106]
    provider_geo_final.loc[mask, "source"] = "manual correction"

    provider_geo_final.to_csv(MAP_CSV, index=False)
    print(f"Resolved {len(provider_geo_final)} providers — saved to {MAP_CSV}")
else:
    provider_geo_final = pd.DataFrame()
    print("No geo data available — run with a valid API key first.")


---
## 2. Data Cleaning & Integration <a id='2-data-cleaning--integration'></a>

### 2.1 RQ1 master table — merge datasets & compute mobilisation rate

In [ ]:
# Data integration
df_final = df_countries.merge(df_europeana, on='iso_code') \
                    .merge(df_exp_filtered, on='iso_code') \
                    .merge(df_wikidata, on='iso_code') \
                    .merge(df_providers, on='iso_code')

df_final['mobilitation_rate_glam'] = (df_final['europeana_providers'] / df_final['glam_count_wikidata']) * 100

formatted_df = df_final.style.format({
    'europeana_total': '{:,.0f}',
    'mobilitation_rate_glam': '{:.2f}%',
    'culture_expenditure_gpd_2022': '{:.2f}%'
})

formatted_df

### 2.2 RQ3 — Provider sample coverage

Coverage statistics for the top-100 provider sample used in classification.

In [ ]:
print("Coverage by top-N providers (% of country total):")
for country, df in providers_by_country.items():
    print(f"  {country}: top-20 → {coverage_at_n(df, 20):.1%}, "
          f"top-50 → {coverage_at_n(df, 50):.1%}, "
          f"top-100 → {coverage_at_n(df, 100):.1%} "
          f"(of {len(df)} total providers)")

print(f"\n{len(top_providers_df)} (country, provider) rows / "
      f"{len(distinct_providers)} distinct providers in the sample")


### 2.3 RQ3 — Classification rules

Regex rules covering institution names and Wikidata descriptions in multiple European languages. The classifier checks in order: Wikidata description → institution name → SPARQL instance_of/field_of_work fallback.

In [ ]:
INSTITUTION_RULES = [
    (r"film archive|cin[ée]math[èe]que|kinemathek|audiovisual archive|\baudiovisual\b|\bcinema\b|cinecitt[aà]|"
     r"moving image|sound\s*(&|and)\s*vision|film museum|film institute", "audiovisual/film archive"),
    (r"\bmusic\b|musical|ethnomusicology|conservator(y|io|oire)|philharmonic|philharmonie|concert hall",
     "audiovisual/film archive"),

    (r"national library|public library|research library|\blibrar", "library/archive"),
    (r"\barchiv|\barquiv|national archives|repositor", "library/archive"),
    (r"bibliotec|bibliothek", "library/archive"),

    (r"natural history museum|science museum|herbarium|botanic|\bzoo\b|planetarium|aquarium|"
     r"geological survey|tropical.{0,15}research", "natural history/science institution"),

    (r"art museum|history museum|national museum|encyclopedic museum|archaeological museum|"
     r"specialized museum|military museum|open-air museum|\bmuseum\b|\bmuseo\b|mus[ée]e|\bmuseu\b|"
     r"museen|\bgallery\b|art collection|photograph|\bcastle\b|\bpalace\b|historical monument|"
     r"epigraphic|archaeolog|archeolog", "art/history museum"),

    (r"broadcaster|television|radio station|newspaper|press agency|publishing house",
     "media/broadcast organization"),

    (r"\buniversity\b|research institute|academy of sciences|research cent(er|re)|\bresearch\b|"
     r"\blaboratory\b|laboratoire", "academic/research institution"),

    (r"government agency|ministry|municipality|public administration|state institution|\bgovernment\b|"
     r"gobierno|ajuntament|gemeente|province of|heritage agency|superintendence|\bcourt\b",
     "government/administrative body"),
]
_compiled_institution_rules = [(re.compile(pat, re.IGNORECASE), name) for pat, name in INSTITUTION_RULES]


def classify_text(text):
    if not text:
        return None
    for pattern, category in _compiled_institution_rules:
        if pattern.search(text):
            return category
    return "other"


def classify_from_sparql(qid):
    info = sparql_info.get(qid, {"instance_of": [], "field_of_work": []})

    fow_category = classify_text(" | ".join(info["field_of_work"]))
    if fow_category not in (None, "other"):
        return fow_category

    io_category = classify_text(" | ".join(info["instance_of"]))
    if io_category not in (None, "other"):
        return io_category

    if not info["field_of_work"] and not info["instance_of"]:
        return None
    return "other"


def classify_provider(name):
    name_category = classify_text(name)
    wd_info = wikidata_resolution.get(name)
    desc_category = classify_text(wd_info.get("description_en")) if wd_info else None

    if name_category not in (None, "other") and desc_category not in (None, "other") and name_category != desc_category:
        # disagreement -- flag for review rather than silently picking one
        return "other", "name_description_conflict"
    if name_category not in (None, "other"):
        return name_category, "provider_name"
    if desc_category not in (None, "other"):
        return desc_category, "wikidata_description"

    if wd_info:
        sparql_category = classify_from_sparql(wd_info.get("qid"))
        if sparql_category not in (None, "other"):
            return sparql_category, "wikidata_sparql_fallback"
        if desc_category == "other" or sparql_category == "other" or name_category == "other":
            return "other", "unmatched"

    if name_category == "other":
        return "other", "unmatched"

    return None, None

classification_rows = []
for name in distinct_providers["provider"]:
    category, source = classify_provider(name)
    classification_rows.append({"provider": name, "provider_category": category, "classification_source": source})

classification_df = pd.DataFrame(classification_rows)
classification_df.to_csv(DATA_DIR / "provider_classification.csv", index=False)
print(classification_df["provider_category"].value_counts(dropna=False))
print()
print(classification_df["classification_source"].value_counts(dropna=False))


### 2.4 RQ3 — Confirmed fixes (hard-coded)

Specific misclassifications identified during manual review, kept in code so they can never be lost due to a stale CSV reload.

In [ ]:
CONFIRMED_FIXES = {
    # Entity-resolution errors (Wikidata matched the wrong entity or a misleading description)
    "Brixiana": "library/archive",  # confirmed via the platform's own description -- a digital library
    "Paul Van Riel": "other",  # individual photographer, not an institution

    # Government heritage-protection agencies mismatched by museum/archaeology keywords
    "Cultural Heritage Agency of the Netherlands": "government/administrative body",
    "Historical Monuments: Regional Conservation": "government/administrative body",
    "Ministry of Culture and Communication, Regional Archaeology Service": "government/administrative body",

    # Archaeology/epigraphic RESEARCH institutes mismatched as museums (the "archaeolog"/"epigraphic"
    # keywords don't distinguish a museum's archaeological collection from a research institute)
    "German Archaeological Institute": "academic/research institution",
    "Epigraphic Database Roma": "academic/research institution",
    "Epigraphic Dabatase Bari": "academic/research institution",  # typo "Dabatase" is in the raw provider name
    "University Institute for Research in Iberian Archeology": "academic/research institution",
    "CISA -Interdipartimental Center for Archaeology": "academic/research institution",

    # "Conservatory" false-friend matches -- not music institutions despite the word
    "National Conservatory of Arts and Crafts": "academic/research institution",  # CNAM, a French engineering university
    "Conservatory of the Gironde Estuary": "government/administrative body",  # environmental conservation body

    # Science/technology museums swept into the generic art/history museum bucket
    "Museon-Omniversum": "natural history/science institution",
    "Technoseum": "natural history/science institution",
    "Zoological Research Museum Koenig": "natural history/science institution",
}


### 2.5 RQ3 — Manual review

Providers classified as `other` or `None` are exported to `data/manual_review.csv`. Previously-filled manual categories are preserved across runs.

In [ ]:
review_rows = []
for name in distinct_providers["provider"]:
    if name in CONFIRMED_FIXES:
        continue
    category, source = classify_provider(name)
    if category in (None, "other"):
        wd = wikidata_resolution.get(name, {})
        item_count = top_providers_df.loc[top_providers_df["provider"] == name, "count"].sum()
        review_rows.append({
            "provider": name, "description_en": wd.get("description_en"),
            "category": category, "source": source, "count": item_count,
        })

new_review_df = pd.DataFrame(review_rows).sort_values("count", ascending=False)

review_path = DATA_DIR / "manual_review.csv"
if review_path.exists():
    existing_df = pd.read_csv(review_path)
    existing_categories = dict(zip(existing_df["provider"], existing_df["manual_category"]))
    new_review_df["manual_category"] = new_review_df["provider"].map(existing_categories).fillna("")
else:
    new_review_df["manual_category"] = ""

new_review_df.to_csv(review_path, index=False)
print(f"{len(new_review_df)} providers need review, {new_review_df['count'].sum():,} total items")
filled = (new_review_df["manual_category"] != "").sum()
print(f"{filled} already have a manual_category from a previous pass")
new_review_df.head(30)


**Review `data/manual_review.csv`**, sorted by `count` descending, save, then reload:

In [ ]:
review_df = pd.read_csv(DATA_DIR / "manual_review.csv")
review_df["manual_category"] = review_df["manual_category"].fillna("")
manual_overrides = {
    row["provider"]: row["manual_category"].strip()
    for _, row in review_df.iterrows()
    if row["manual_category"].strip()
}
print(f"{len(manual_overrides)} manual overrides loaded from CSV")
print(f"{len(CONFIRMED_FIXES)} confirmed fixes hard-coded")


def classify_provider_final(name):
    if name in CONFIRMED_FIXES:
        return CONFIRMED_FIXES[name], "confirmed_fix"
    if name in manual_overrides:
        return manual_overrides[name], "manual_override"
    return classify_provider(name)


### 2.6 RQ3 — Aggregate category share by country

In [ ]:
final_categories = {name: classify_provider_final(name)[0] for name in distinct_providers["provider"]}

top_providers_df["provider_category"] = top_providers_df["provider"].map(final_categories)
top_providers_df["provider_category"] = top_providers_df["provider_category"].fillna("unresolved")

category_by_country = (
    top_providers_df.groupby(["country", "provider_category"])["count"]
    .sum()
    .unstack(fill_value=0)
)

# bring in the TRUE total per country from step 1 (not just the sum of sampled providers)
category_by_country = category_by_country.merge(
    totals_df.set_index("country")["total_items"], left_index=True, right_index=True
)

sample_total = category_by_country.drop(columns="total_items").sum(axis=1)

# share of each country's TRUE total (will sum to LESS than 100% -- the gap is the unsampled long tail)
category_share = category_by_country.drop(columns="total_items").div(
    category_by_country["total_items"], axis=0
)

# how much of the true total the sample actually covers, per country
category_share["coverage"] = sample_total / category_by_country["total_items"]

category_share = (category_share * 100).round(2)
category_share = category_share.reset_index()

category_share.to_csv(DATA_DIR / "complete_df.csv", index=False)
category_share

### 2.7 RQ3 — Full per-provider audit table

In [ ]:
audit_rows = []
for name in distinct_providers["provider"]:
    category, source = classify_provider_final(name)
    count = top_providers_df.loc[top_providers_df["provider"] == name, "count"].sum()
    country = distinct_providers.loc[distinct_providers["provider"] == name, "country"].iloc[0]
    audit_rows.append({
        "provider": name,
        "country": country,
        "count": count,
        "provider_category": category if category else "unresolved",
        "classification_source": source if source else "unresolved",
    })

cat_df = pd.DataFrame(audit_rows).sort_values(["country", "count"], ascending=[True, False])
cat_df.to_csv(DATA_DIR / "cat_full.csv", index=False)

pd.set_option("display.max_rows", None)
cat_df.head(30)


### 2.8 RQ2 — Quality & openness summary

Two components make up RQ2:

- **Quality**: share of items meeting contentTier (2/3/4) and metadataTier (A/B/C) thresholds.
- **Openness**: share of items under open, restricted, or permission-required licenses.

In [ ]:
def quality_summary(raw):
    rows = []
    for country, data in raw.items():
        total = data["totalResults"]
        ct = facet_dict(data, "contentTier")
        mt = facet_dict(data, "metadataTier")
        content_ok = sum(v for k, v in ct.items() if k in QUALITY_CONTENT_TIERS)
        metadata_ok = sum(v for k, v in mt.items() if k in QUALITY_METADATA_TIERS)
        rows.append({
            "country": country, "total_items": total,
            "pct_contentTier_2plus": round(100 * content_ok / total, 1),
            "pct_contentTier_0": round(100 * ct.get("0", 0) / total, 1),
            "pct_metadataTier_ABC": round(100 * metadata_ok / total, 1),
            "pct_metadataTier_0": round(100 * mt.get("0", 0) / total, 1),
        })
    return pd.DataFrame(rows).sort_values("pct_contentTier_2plus", ascending=False)


def contenttier_breakdown(raw):
    rows = []
    for country, data in raw.items():
        total = data["totalResults"]
        ct = facet_dict(data, "contentTier")
        row = {"country": country, "total_items": total}
        for tier in ["0", "1", "2", "3", "4"]:
            row[f"pct_tier_{tier}"] = round(100 * ct.get(tier, 0) / total, 1)
        rows.append(row)
    return pd.DataFrame(rows).sort_values("pct_tier_4", ascending=False)


def metadatatier_breakdown(raw):
    rows = []
    for country, data in raw.items():
        total = data["totalResults"]
        mt = facet_dict(data, "metadataTier")
        row = {"country": country, "total_items": total}
        for tier in ["0", "A", "B", "C"]:
            row[f"pct_metadataTier_{tier}"] = round(100 * mt.get(tier, 0) / total, 1)
        rows.append(row)
    return pd.DataFrame(rows).sort_values("pct_metadataTier_C", ascending=False)


def openness_summary(reusability_results):
    rows = []
    for country, counts in reusability_results.items():
        total = sum(counts.values())
        rows.append({
            "country": country,
            "pct_open": round(100 * counts["open"] / total, 1) if total else None,
            "pct_restricted": round(100 * counts["restricted"] / total, 1) if total else None,
            "pct_permission": round(100 * counts["permission"] / total, 1) if total else None,
        })
    return pd.DataFrame(rows)


if raw_results:
    quality_df = quality_summary(raw_results)
    tier_breakdown_df = contenttier_breakdown(raw_results)
    metadata_breakdown_df = metadatatier_breakdown(raw_results)
    if reusability_results:
        openness_df = openness_summary(reusability_results)
        rq2_df = quality_df.merge(openness_df, on="country")
        rq2_df.to_csv(DATA_DIR / "rq2_quality_openness.csv", index=False)
    else:
        rq2_df = quality_df.copy()
    display(quality_df)
else:
    print("raw_results is empty — re-run section 1.5 with a valid API key.")


### 2.9 RQ2 — Provider concentration

How much of each country's total item volume is concentrated in its top providers?

In [ ]:
def provider_concentration(raw, top_n=5):
    rows = []
    for country, data in raw.items():
        total = data["totalResults"]
        dp = facet_dict(data, "DATA_PROVIDER")
        sorted_providers = sorted(dp.items(), key=lambda x: -x[1])
        top_counts = [c for _, c in sorted_providers[:top_n]]
        top1_pct = round(100 * top_counts[0] / total, 1) if top_counts else 0
        topn_pct = round(100 * sum(top_counts) / total, 1) if top_counts else 0
        rows.append({
            "country": country,
            "pct_top1_provider": top1_pct,
            f"pct_top{top_n}_providers": topn_pct,
            "n_providers": len(dp),
        })
    return pd.DataFrame(rows).sort_values("pct_top1_provider", ascending=False)


if raw_results:
    concentration_df = provider_concentration(raw_results)
    display(concentration_df)


---
## 3. Visualizations & Analysis <a id='3-visualizations--analysis'></a>

### Shared color palette

All country-level visualizations use this single color mapping for consistency.

In [ ]:
# ── Country colors (reused across all visualizations) ────────────────────────
COUNTRY_COLORS = {
    "netherlands": "#a180ad",
    "france":      "#1f7f95",
    "portugal":    "#f4a64e",
    "italy":       "#90BE6D",
    "germany":     "#feda15",
    "spain":       "#bb521f",
}

# Title-case variant for visualizations that use full country names (RQ2)
COUNTRY_COLORS_TC = {k.capitalize(): v for k, v in COUNTRY_COLORS.items()}

# ISO-code variant for RQ1 visualizations (uses iso_code column)
ISO_TO_LOWER = {
    "IT": "italy", "DE": "germany", "NL": "netherlands",
    "PT": "portugal", "ES": "spain", "FR": "france",
}
COUNTRY_COLORS_ISO = {iso: COUNTRY_COLORS[lower] for iso, lower in ISO_TO_LOWER.items()}

print("Country color palette:")
for name, color in COUNTRY_COLORS.items():
    print(f"  {name:<15} {color}")


## Abstract
This quantitative question investigates whether the volume of preserved and shared digital cultural heritage on Europeana reflects the physical density of Galleries, Libraries, Archives, and Museums (GLAMs) or the scale of public cultural investment (% of GDP via Eurostat). We merge SPARQL queries from Wikidata and Europeana with offline administrative census data across six European nations. Our analysis uncovers a critical paradox: government funding does not linearly correlate with digital volume.

### Research questions


*  Is there a statistically significant correlation between government cultural expenditure (% of GDP) and the total volume of digitized cultural heritage shared on Europeana?
* To what extent does a nation's physical GLAM density predict the number of active, data-contributing providers on Europeana?

### Sources


1.   **Europeana SPARQL API / Portal Counts**: Total digital items shared per country (`europeana_total`).
2.   **Europeana Active Providers Database**: Number of active data providers contributing (`europeana_providers`).
3. **Eurostat (COFOG)**: Public expenditure on cultural services as a percentage of national GDP (`culture_expenditure_gpd_2022`).
4. **Wikidata Query Service (SPARQL)**: Baseline count of GLAM institutions mapped on the Semantic Web (`glam_count_wikidata`).
5. **Statistics Portugal (INE / Official Census)**: Offline administrative registry used as a validation benchmark to resolve open-data under-sampling (`portugal_glam_census.csv`).


### 3.1 RQ1 — GLAM mobilisation rate by country

In [ ]:
# Visualization Stage 1 - Mobilitation Rate Glam
df_support = df_final[['iso_code', 'mobilitation_rate_glam']].copy()
df_support['Active on Europeana (%)'] = df_support['mobilitation_rate_glam']
df_support['Offline GLAMs (%)'] = 100 - df_support['mobilitation_rate_glam']

df_support = df_support.sort_values(by='mobilitation_rate_glam', ascending=False)

df_tidy = df_support.melt(
    id_vars=['iso_code', 'mobilitation_rate_glam'],
    value_vars=['Active on Europeana (%)', 'Offline GLAMs (%)'],
    var_name='Status',
    value_name='Percentage'
)

fig1 = px.bar(
    df_tidy,
    x='iso_code',
    y='Percentage',
    color='Status',
    title="Stage 1: The Mobilization Gap: distribution of active europeana providers",
    labels={
        'iso_code': 'Country',
        'Percentage': 'Share of Physical GLAMs (%)',
        'Status': 'Institutional Status'
    },
    color_discrete_map={
        'Active on Europeana (%)': '#5A5A5A',
        'Offline GLAMs (%)': '#AACAE0'
    }
)

fig1.update_traces(
    texttemplate='',
    hoverinfo='none'
)

df_active = df_tidy[df_tidy['Status'] == 'Active on Europeana (%)']

fig1.add_trace(
    go.Scatter(
        x=df_active['iso_code'],
        y=df_active['Percentage'],
        text=df_active['Percentage'].map('{:.1f}%'.format),
        mode='text',
        textposition='top center',
        textfont=dict(size=12, color='black'),
        showlegend=False,
        hoverinfo='none'
    )
)

fig1.update_layout(
    template="plotly_white",
    height=600,
    width=800,
    barmode='stack',
    hovermode=False,
    xaxis=dict(showgrid=False),
    yaxis=dict(
        showgrid=True,
        gridcolor="lightgray",
        ticksuffix="%",
        range=[0, 108]
    ),
    legend=dict(
        orientation="h",
        yanchor="bottom",
        y=1.02,
        xanchor="right",
        x=1
    )
)

fig1.show()

This normalized 100% stacked bar chart explores the proportion of active Europeana data providers relative to the broader baseline of physical GLAM institutions mapped via Wikidata across selected countries. By removing absolute volume differences, the visual distribution highlights a notable disparity: across all observed territories, over 89% of mapped institutions do not currently appear as active contributors to Europeana's Linked Data infrastructure, gettin lost in the sea. This observational pattern suggests that a country's institutional density does not necessarily reflect its active online participation rate.

### 3.2 RQ1 — Cultural expenditure vs. Europeana digital volume

In [ ]:
# Visualization Stage 2 - Culture expenditure
x_vals = df_final['culture_expenditure_gpd_2022'].values
y_vals = df_final['europeana_total'].values

m = np.sum(x_vals * y_vals) / np.sum(x_vals**2)
q = 0

x_line = np.linspace(0, x_vals.max() + 0.05, 100)
y_line = m * x_line

fig2 = px.scatter(
    df_final,
    x="culture_expenditure_gpd_2022",
    y="europeana_total",
    text="iso_code",
    color="iso_code",
    color_discrete_map=COUNTRY_COLORS_ISO,
    title="Stage 2: Relationship between public funding and total shared items in Europeana",
    labels={
        "culture_expenditure_gpd_2022": "Public Funding in Culture (% of GDP)",
        "europeana_total": "Total Objects Shared on Europeana"
    },
    hover_data={
        "europeana_total": ":,.0f",
        "culture_expenditure_gpd_2022": False,
        "iso_code": False
    }
)

fig2.add_trace(
    go.Scatter(
        x=x_line,
        y=y_line,
        mode="lines",
        name="Global Trendline (OLS)",
        line=dict(dash="dash", color="#FF4B4B", width=2.5),
        showlegend=False,
        hoverinfo="skip"
    )
)

fig2.update_layout(
    template="plotly_white",
    height=550,
    width=750,
    showlegend=False,
    xaxis=dict(
        showgrid=True,
        gridcolor="lightgray",
        ticksuffix="%",
        range=[x_vals.min() - 0.08, x_vals.max() + 0.08]
    ),
    yaxis=dict(
        showgrid=True,
        gridcolor="lightgray",
        tickformat=","
    )
)

fig2.update_traces(
    selector=dict(mode="markers+text"),
    textposition="top center",
    textfont=dict(size=12, family="sans-serif", color="black")
)


fig2.show()

To explore whether higher public funding in culture (% of GDP) aligns with higher volumes of shared digital objects, we map the two variables using a scatter plot. The trendline serves to understand the missing positive correlation we expected. We observe a wide dispersion of data points, suggesting a null correlation. For instance, France (1.4% GDP) and the Netherlands (1.1% GDP) display noticeably different digital object volumes (4.7M vs. 9.2M objects) despite relatively high spending levels. This distribution indicates that macroeconomic funding percentages alone do not account for the observed variance in online artifact accumulation, suggesting that the funding are still not oriented towards european level sharing of digitalized data.

### 3.3 RQ1 — Funding × mobilisation × digital volume (bubble chart)

In [ ]:
# Visualization stage 3a- Relationship between investment, glam mobilitation rate, europeana total item for each selected countries
fig3 = px.scatter(
    df_final,
    x="mobilitation_rate_glam",
    y="culture_expenditure_gpd_2022",
    size="europeana_total",
    color="iso_code",
    color_discrete_map=COUNTRY_COLORS_ISO,
    text="iso_code",
    hover_name="name_en",

    hover_data={
        "europeana_total": ":,",
        "mobilitation_rate_glam": ":.2f",
        "culture_expenditure_gpd_2022": ":.2f%",
        "iso_code": False
    },

    title="Stage 3a: European(a) Digital Heritage: relationship between investment,<br> glam mobilitation rate and total shared items</br>",
    labels={
        "mobilitation_rate_glam": "GLAM Mobilitation Rate (% of physical institutions active online)",
        "culture_expenditure_gpd_2022": "Public Funding in Culture (% of GDP)",
        "europeana_total": "Total Objects in Europeana",
    },
    size_max=65
)

fig3.update_layout(
    template="plotly_white",
    height=800,
    width=750,
    showlegend=False,
    margin=dict(t=80),
    xaxis=dict(
        showgrid=True,
        gridcolor="lightgray",
        ticksuffix="%"
    ),
    yaxis=dict(
        showgrid=True,
        gridcolor="lightgray",
        ticksuffix="%"
    )
)

fig3.update_traces(
    textposition="top center",
    textfont=dict(size=12, family="sans-serif", color="black")
)

fig3.show()



This multivariate bubble chart simultaneously explores public funding (Y-axis), observed mobilization rate (X-axis), and total digital volume (bubble size). In this initial distribution, calculated using crowdsourced Wikidata counts as the baseline denominator, Portugal (PT) appears positioned at a relatively high mobilization rate (8.70%). We note that the baseline Wikidata count for Portugal is noticeably smaller (460 institutions) than that of other nations, prompting an exploratory sensitivity check on how denominator definitions influence comparative metrics.

### 3.4 RQ1 — Corrected bubble chart with Portugal administrative census

In [ ]:
# Stage 3b: Update with data from portugal GLAM census
df_ine = pd.read_csv('data/portugal_glam_census.csv')
pt_real_glam_total = df_ine['Value'].sum()
df_final.loc[df_final['iso_code'] == 'PT', 'glam_count_wikidata'] = pt_real_glam_total

df_final['mobilitation_rate_glam'] = (df_final['europeana_providers'] / df_final['glam_count_wikidata']) * 100

fig3 = px.scatter(
    df_final,
    x="mobilitation_rate_glam",
    y="culture_expenditure_gpd_2022",
    size="europeana_total",
    color="iso_code",
    color_discrete_map=COUNTRY_COLORS_ISO,
    text="iso_code",
    hover_name="name_en",
    hover_data={
        "europeana_total": ":,",
        "mobilitation_rate_glam": ":.2f",
        "culture_expenditure_gpd_2022": ":.2f%",
        "iso_code": False
    },
    title="Stage 3b: European(a) Digital Heritage: relationship between investment, <br> glam mobilitation rate (updated) and total shared items </br>",
    labels={
        "mobilitation_rate_glam": "GLAM Mobilitation Rate (% of physical institutions active online)",
        "culture_expenditure_gpd_2022": "Public Funding in Culture (% of GDP)",
        "europeana_total": "Total Objects in Europeana",
    },
    size_max=65
)

fig3.update_layout(
    template="plotly_white",
    height=800,
    width=750,
    showlegend=False,
    margin=dict(t=80),
    xaxis=dict(
        showgrid=True,
        gridcolor="lightgray",
        ticksuffix="%"
    ),
    yaxis=dict(
        showgrid=True,
        gridcolor="lightgray",
        ticksuffix="%"
    )
)

fig3.update_traces(
    textposition="top center",
    textfont=dict(size=12, family="sans-serif", color="black")
)

fig3.show()



To better observe the correlation, we replace Portugal's Wikidata baseline (460 institutions) with administrative census data from Statistics Portugal (INE: 3,395 institutions). Maintaining an identical spatial grid (range=[-0.5, 12.5]), we observe a noticeable horizontal shift: Portugal's observed mobilization rate adjusts from 8.70% to 1.18%. This visual variance help us better investigate the correlation: a positive correlation in term of mobilitation rate of GLAM institution on Europeana is shown. Spain, unexpected negative, has a smaller number of total europeana item, showing the investment is not directly correlated to the % of investments. France stand outside this correlation, the country invest more then all the other observed countries but its mobilitation rate shows that a small only of GLAM institution shares data on Europeana. Italy proves to be an expected negative: small investment, small distrubution of providers over total GLAM institutions and, consequentially, small number of total items on Europeana.

### 3.5 RQ3 — Full provider list coverage by country

How many providers are needed to cover 80 %, 90 %, and 95 % of each country's total Europeana item volume.

In [ ]:
summary_rows = []
for country, df in providers_by_country.items():
    sorted_df = df.sort_values("count", ascending=False)
    total = sorted_df["count"].sum()
    cumulative = sorted_df["count"].cumsum()
    summary_rows.append({
        "country": country,
        "total_providers": len(df),
        "n_for_80pct": int((cumulative / total < 0.80).sum()) + 1,
        "n_for_90pct": int((cumulative / total < 0.90).sum()) + 1,
        "n_for_95pct": int((cumulative / total < 0.95).sum()) + 1,
        "pct_top20": round(coverage_at_n(df, 20) * 100, 1),
        "pct_top100": round(coverage_at_n(df, 100) * 100, 1),
    })
pd.DataFrame(summary_rows)


### 3.6 RQ2 — Provider concentration grid

* **pct_top1_provider** — what pct of all items in the country come from the single biggest provider? 
* **pct_top5_providers** -  what pct of all items are accounted for by the top 5 providers combined? This tells you how "thick" the tail is: if top8_share is low (e.g. 30%), most of the country's items are spread across many small/medium providers beyond the visible top 8. If it's high (e.g. 90%), the top 5 basically are the country's collection.


In [ ]:
import plotly.graph_objects as go
from plotly.subplots import make_subplots

# Order countries by concentration so the grid tells a story (most concentrated first)
plot_df = concentration_df.sort_values("pct_top1_provider", ascending=False).reset_index(drop=True)

fig = make_subplots(
    rows=2, cols=3,
    specs=[[{"type": "treemap"}]*3, [{"type": "treemap"}]*3],
    subplot_titles=plot_df["country"].tolist(),
    horizontal_spacing=0.03,
    vertical_spacing=0.12,
)

segment_colors = ["#203464", "#4b74a0", "#679e91"]  # dark = most concentrated, light = most distributed

for idx, row in plot_df.iterrows():
    r = idx // 3 + 1
    c = idx % 3 + 1

    top1 = row["pct_top1_provider"]
    top5_minus_1 = row["pct_top5_providers"] - top1
    others = 100 - row["pct_top5_providers"]

    fig.add_trace(
        go.Treemap(
            labels=["Top 1", "Rank 2–5", "Other"],
            parents=["", "", ""],
            values=[top1, top5_minus_1, others],
            marker=dict(colors=segment_colors, line=dict(width=1, color="white")),
            texttemplate="%{label}<br>%{value:.0f}%",
            textfont=dict(size=13, color="white"),
            hovertemplate="%{label}: %{value:.1f}%<extra></extra>",
        ),
        row=r, col=c,
    )

# Manual shared legend via dummy bar traces (treemaps don't support a clean shared legend natively)
for label, color in zip(["Top 1 provider", "Rank 2–5", "Other providers"], segment_colors):
    fig.add_trace(go.Bar(x=[None], y=[None], marker_color=color, name=label, showlegend=True))

fig.update_layout(
    title=dict(text="Provider concentration by country", x=0.02, font=dict(size=20)),
    height=650, width=1050,
    legend=dict(orientation="h", yanchor="bottom", y=-0.08, x=0.3),
    margin=dict(t=90, b=60, l=20, r=20),
    paper_bgcolor="white",
    font=dict(family="Arial, sans-serif"),
)

# Style subplot titles (country names) consistently
for annotation in fig["layout"]["annotations"]:
    annotation["font"] = dict(size=15, color="#333333")

fig.show()

### 3.7 RQ2 — Geographic distribution of top-20 providers

### Geographic distribution of top-20 providers

To map where digitization activity concentrates within each country, we take each 
country's top 20 providers by item count and attempt to resolve a city-level location 
for each, using two sources in sequence:

1. **Europeana's own Organization entity profile** (direct address/geo data, curated by Europeana).
2. **Wikidata**, as a fallback for providers without a usable Europeana geo — first via
   the Wikidata URI already cross-referenced in Europeana's entity record where available,
   otherwise via a name-based search.

Providers that resolve through neither path are handled separately below.

In [ ]:
import plotly.express as px
import numpy as np

provider_geo_final["count_log"] = np.log10(provider_geo_final["count"] + 1)

# Uses shared COUNTRY_COLORS_TC defined at start of section 3

fig = px.scatter_geo(
    provider_geo_final,
    lat="latitude",
    lon="longitude",
    color="country",
    size="count_log", size_max=25,
    hover_name="provider",
    hover_data={"city": True, "count": ":,","count_log": False, "source": True, "latitude": False, "longitude": False},
    projection="natural earth",
    scope="europe",
    title="Top providers by location and item volume",
    color_discrete_map=COUNTRY_COLORS_TC
)

fig.update_layout(
    height=650, width=950,
    legend_title="Country",
    margin=dict(l=10, r=10, t=60, b=10),
)
fig.update_geos(
    showcountries=True, countrycolor="#CCCCCC",
    showland=True, landcolor="#F5F5F5",
    showocean=True, oceancolor="#EAF2F8",
)

fig.show()

## 10. Visualization -- sunburst, country contribution -> provider typology

**Level 1 (inner ring):** each country's share of item volume, among the six countries
studied here (not a global Europeana share -- only these six countries and their top-100
sampled providers each are represented).

**Level 2 (outer ring, click a country to drill in):** that country's provider-typology
composition -- the same category breakdown as before, now explorable per country instead of
compared all at once in a single bar chart.


### 3.8 RQ3 — Institution type sunburst by country

In [ ]:
# Uses shared COUNTRY_COLORS defined at start of section 3
CATEGORY_COLORS = {
    "audiovisual/film archive": "#C2528C",              # rose/magenta
    "art/history museum": "#5C6BC0",                     # indigo
    "natural history/science institution": "#2A9D8F",   # teal-green
    "library/archive": "#6FA8DC",                        # soft sky blue
    "academic/research institution": "#7E6B8F",          # dusty violet-gray
    "media/broadcast organization": "#D4A24C",           # muted gold (deliberately duller than Portugal's brighter orange)
    "government/administrative body": "#8C4A4A",         # muted brick red
    "other": "#B9B9B9",                                  # neutral gray
    "unresolved": "#7A7A7A",                             # darker neutral gray
}
provider_counts_by_country = distinct_providers.groupby("country").size()
total_providers = len(distinct_providers)

sunburst_country = cat_df.groupby("country")["count"].sum().reset_index()
sunburst_cat = cat_df.groupby(["country", "provider_category"])["count"].sum().reset_index()

ids, labels, parents, values, colors, hover_text = [], [], [], [], [], []

# level 1 -- countries, colored individually, hover shows % of PROVIDERS (not % of items)
for _, row in sunburst_country.iterrows():
    country = row["country"]
    item_count = row["count"]
    n_providers = provider_counts_by_country.get(country, 0)
    provider_pct = n_providers / total_providers * 100

    ids.append(country)
    labels.append(country.capitalize())
    parents.append("")
    values.append(item_count)
    colors.append(COUNTRY_COLORS.get(country, "#CCCCCC"))
    hover_text.append(
        f"<b>{country.capitalize()}</b><br>{item_count:,} items<br>{provider_pct:.1f}% of all providers"
    )

# level 2 -- categories within each country, hover shows % of THAT COUNTRY's items
for _, row in sunburst_cat.iterrows():
    country, cat, count = row["country"], row["provider_category"], row["count"]
    country_total = sunburst_country.loc[sunburst_country["country"] == country, "count"].iloc[0]
    cat_pct_of_country = count / country_total * 100

    ids.append(f"{country}-{cat}")
    labels.append(cat)
    parents.append(country)
    values.append(count)
    colors.append(CATEGORY_COLORS.get(cat, "#D1D5DB"))
    hover_text.append(
        f"<b>{cat}</b><br>{count:,} items<br>{cat_pct_of_country:.1f}% of {country.capitalize()}"
    )

fig = go.Figure(go.Sunburst(
    ids=ids,
    labels=labels,
    parents=parents,
    values=values,
    branchvalues="total",
    marker=dict(colors=colors),
    customdata=hover_text,
    hovertemplate="%{customdata}<extra></extra>",
    insidetextorientation="horizontal",
))

fig.update_layout(height=750, margin=dict(t=30, l=0, r=0, b=0))
fig.show()

### 3.9 RQ2 — Content tier & metadata tier quality by country

To assess the quality of the records contributed by different countries, Europeana classifies each item according to two complementary quality frameworks: Metadata Tier and Content Tier. Together, these indicators evaluate how well an object is described and how useful its associated digital representation is for users.

The **Content Tier** measures content quality and resuability by taking into account not just the quality of the digital resources, but also the rights statements and licenses applied to them: the fewer copyright restrictions placed on digital objects, the higher their potential for reuse and the higher value to end users. The classification ranges from 0 to 4, with Europeana considering CT2-CT4 as meeting its minimum publishing quality requirements, while CT0 and CT1 do not. the aggregated metric **pct_contentTier_2plus** represents the proportion of records that satisfy these quality criteria. However only tiers 3 and 4 add a rights requirement on top of technical quality, meaning that they're only reachable if the object carries a rights statement that allows reuse, for this reason the function **contenttier_breakdown()** further decomposes the distribution by individual tier, which is what allows checking whether a country's tier-2+ score is coming from rights-agnostic tier 2 content, or from
rights-gated tier 3/4 content, and therefore whether openness should be expected to track it.

The **Metadata Tier** measures the completeness and richness of the descriptive metadata provided with each record scored across three criteria
(language tagging, enabling elements, and contextual class links), with the overall tier determined by the *lowest*-scoring criterion. Europeana's documentation formally defines three levels, A through C, each with increasing thresholds. A "0" value also appears in the live API data and in Europeana's own example datasets, functioning as the equivalent of contentTier's tier 1: records that don't meet even the minimum bar for tier C. Our **pct_metadataTier_AB** metric captures the two tiers Europeana defines as meeting its quality standard.

In [ ]:
import plotly.graph_objects as go

order = rq2_df.sort_values("pct_contentTier_2plus", ascending=False)["country"]
plot_df = rq2_df.set_index("country").loc[order]

# --- Quality chart (grouped bars) ---
fig_quality = go.Figure()
fig_quality.add_trace(go.Bar(x=plot_df.index, y=plot_df["pct_contentTier_2plus"],
                              name="Content tier 2+", marker_color="#4C72B0"))
fig_quality.add_trace(go.Bar(x=plot_df.index, y=plot_df["pct_metadataTier_ABC"],
                              name="Metadata tier A/B/C", marker_color="#DD8452"))
fig_quality.update_layout(
    barmode="group",
    title="Quality: content & metadata tier compliance",
    yaxis_title="% of items",
    height=500, width=600,
)
fig_quality.show()

In [ ]:
import plotly.graph_objects as go

order = rq2_df.sort_values("pct_open", ascending=False)["country"]
plot_df = rq2_df.set_index("country").loc[order]

fig_openness = go.Figure()
fig_openness.add_trace(go.Bar(x=plot_df.index, y=plot_df["pct_open"],
                               name="Open", marker_color="#31A354"))
fig_openness.add_trace(go.Bar(x=plot_df.index, y=plot_df["pct_restricted"],
                               name="Restricted", marker_color="#FDAE6B"))
fig_openness.add_trace(go.Bar(x=plot_df.index, y=plot_df["pct_permission"],
                               name="Permission", marker_color="#DE2D26"))
fig_openness.update_layout(
    barmode="stack",
    title="Openness: license/reusability breakdown (ordered by openness)",
    yaxis_title="% of items",
    height=500, width=700,
)
fig_openness.show()

In [ ]:
from plotly.subplots import make_subplots
import plotly.graph_objects as go

content_colors = {
    "pct_tier_0": "#E8E8E8",
    "pct_tier_1": "#C6DBEF",
    "pct_tier_2": "#6BAED6",
    "pct_tier_3": "#2171B5",
    "pct_tier_4": "#08306B",
}
metadata_colors = {
    "pct_metadataTier_0": "#E8E8E8",
    "pct_metadataTier_A": "#FDD0A2",
    "pct_metadataTier_B": "#FD8D3C",
    "pct_metadataTier_C": "#A63603",
}

# Same country order on both subplots -- sort by a meaningful reference,
# e.g. content tier 4 share, so the row order tells its own story
order = tier_breakdown_df.sort_values("pct_tier_4", ascending=True)["country"]  # ascending so highest ends up at top in a horizontal bar
ct = tier_breakdown_df.set_index("country").loc[order]
mt = metadata_breakdown_df.set_index("country").loc[order]

fig = make_subplots(
    rows=1, cols=2,
    subplot_titles=("Content tier composition", "Metadata tier composition"),
    shared_yaxes=True,
    horizontal_spacing=0.08,
)

for col, color in content_colors.items():
    tier_label = col.replace("pct_tier_", "Content – Tier ")
    fig.add_trace(
        go.Bar(y=ct.index, x=ct[col], orientation="h",
               name=tier_label, marker_color=color, legendgroup="content"),
        row=1, col=1
    )

for col, color in metadata_colors.items():
    tier_label = col.replace("pct_metadataTier_", "Metadata – Tier ")
    fig.add_trace(
        go.Bar(y=mt.index, x=mt[col], orientation="h",
               name=tier_label, marker_color=color, legendgroup="metadata"),
        row=1, col=2
    )

fig.update_xaxes(title_text="% of items", row=1, col=1)
fig.update_xaxes(title_text="% of items", row=1, col=2)

fig.update_layout(
    barmode="stack",
    height=500, width=1150,
    title_text="Content vs. metadata tier composition, by country",
    legend=dict(title="Tier", tracegroupgap=10),
)

fig.show()

### Findings

**Quality (Europeana Quality Score).**

* The **Netherlands** leads on content tier: 87.5% of items reach content tier 2+, but its metadata score (85.6% A/B/C) is not the highest in the group, meanign that a mature digitization pipeline (Naturalis, Rijksmuseum) probably translates into strong technical/rights quality more than into top-tier descriptive metadata. 
* **Portugal** and **Germany** actually lead on metadata (97.6% and 94.8% A/B/C respectively), while sitting far apart on content tier (59.1% vs. 76.2%), a first sign that these two quality dimensions move independently rather than together. 
* **Italy** and **France** sit at the bottom for content tier (44.3% and 41.4%), even though both reach comparatively high metadata scores (86.4% and 70.7%) — France in particular combines a middling metadata score with the highest share of content tier 0 in the sample (17.4%), pointing to weak *technical/rights* quality rather than weak *descriptive* quality.

**Openness (license/reusability).**

* **Portugal** (97.7% open) and the **Netherlands** (76.1% open) are the most open. * **Italy** is the extreme opposite: only 11.8% open items against a striking 56.9% requiring explicit permission, the highest "permission" share in the dataset. 
* **France** is also low on openness (11.3%), but with a different pattern: it leans "restricted" (63.3%) rather than "permission-required."

**Quality and openness are not correlated**

Germany combines the second-highest metadata score in the group (94.5% A/B/C) with medium-low openness (28.3%). Italy is an even starker case: 86.4% of its items meet the metadata quality bar, yet only 11.8% are openly licensed — a near-90-point gap between how well items are *described* and how freely they can be *reused*. Portugal is the mirror image: excellent metadata (97.5%) paired with the highest openness in the sample (97.7%). Since content tier is *partly* tied to rights openness at Europeana's higher tiers (tier 3/4 require a reusable rights statement), some content-tier/openness overlap is expected by design — but metadata tier has no such built-in link, so cases like Italy and Germany are genuine empirical findings: institutions can invest heavily in descriptive richness while keeping their licensing restrictive. This supports reading technical/descriptive quality and openness as reflecting different institutional logics — digitization capacity and cataloguing effort on one side, policy choices around intellectual property on the other (e.g. Italy's high "in copyright" share likely reflects a more protective museum/archive system).


**pct_metadataTier_ABC** is an aggregate that could hide where the mass sits. A country could hit a high ABC score almost entirely via tier A (the loosest bar) while another hits it mostly via tier C (the strictest) — those are very different quality profiles that the aggregate alone can't distinguish, exactly like the tier-2-vs-3/4 issue you caught for content tier.

!! Right now these are just national averages, but national averages can mean two different things:
* distributed pattern: many institutions in a country converge independently in similar quality/licensing practices and this can genuinly reflect a national policy 
* concentrated pattern: one or two huge providers (e.g. a national library dumping millions of low-tier newspaper scans) single-handedly drag the country average down, while most other institutions might actually score well.